# Supa Strikers — P3: referee pointing gestures in the broadcast (Kaggle, GPU)

When play stops (a foul, offside, the ball going out), the referee raises an arm and points: up,
or towards the restart. This notebook looks for that gesture **only in the broadcast**, where the
referee is close enough to see the arms, and **only inside the windows P2 selected** around labeller
v6's stoppages and restarts (`outputs/referee/broadcast_scan_ranges.csv`, about 30k frames ≈ 20 min,
not the whole video).

**Pipeline**
1. **People:** Ultralytics **YOLO11m** person detection on every frame of every scan range. Pose on the
   whole frame does not work here: in wide broadcast shots players are ~70–90 px tall and YOLO11-pose
   finds almost none (0 at 640 px, 1 at 1280 px on the sample frame), while the detector finds them all.
2. **Who is the referee** (no hand labels): the torso colour of each person (centre of the box,
   pitch-green removed) is clustered with k-means on a calibration sample. The
   referee cluster is the one present in the most ranges with **≈1 person per frame**; teams show
   several. The assistant referees wear the same kit, so their flag/arm signals count too. Montages
   of each cluster are saved for checking; `REF_CLUSTER` overrides the choice.
3. **Pose on the referee only:** each referee box is cropped with padding for the arms, upscaled to
   ~384 px tall and run through **YOLO11m-pose** (on the sample frame: confidence 0.93, arm keypoints ≥ 0.99).
   **Pointing arm**, from those keypoints: upper arm raised ≥ `ARM_ELEV_MIN`° from the
   torso, elbow straight ≥ `ARM_EXT_MIN`°, hand ≥ `HAND_ELEV_MIN`° from the torso. A running arm swing
   has a bent elbow and stays low. Direction = up / left / right **in the image**.
4. **Events:** runs of pointing frames (holes ≤ `GAP_FRAMES`) lasting ≥ `MIN_HOLD_S`. Each event is
   tied to its P2 window(s), v6 stretch `d`, offset from the stoppage/restart and the tactical frame.
   Slow motion (consecutive frames barely change) is measured per frame, so replay events can be flagged.

**Inputs** (Add Input → your datasets; found anywhere under `/kaggle/input` by file name):
`BAR-ATH_broadcast.mp4` and a small dataset with `broadcast_scan_ranges.csv` + `broadcast_windows.csv`
(from `python scripts/build_broadcast_windows.py`).
**Settings:** Accelerator *GPU T4* (one GPU is enough), Internet *on* (pip `ultralytics` + YOLO11 weights;
or add the `.pt` as a dataset and point `POSE_MODEL` at it).

**How to run:** `RUN_MODE = "selftest"` (synthetic video, a fake pose model, ~1 min, no data) →
`"smoke"` (8 ranges spread over the match, ~5 min: check the referee-cluster montage and the review
clips) → `"full"` → *Save Version → Save & Run All* (~20–30 min on a T4). Chunks are saved per range,
so an interrupted run resumes; add a previous version's output as an input to reuse them.

**Outputs** (`/kaggle/working/referee/`): `referee_events.csv`, `referee_windows.csv`,
`referee_stretches.csv`, `referee_frames.parquet` (per referee detection, with keypoint features),
`referee_summary.json`, `ref_clusters.png`, `clips/*.mp4` (events with skeleton overlay).

**Compliance (RULES.md):** pretrained pose model, unsupervised referee identification, fixed geometric
thresholds; no hand labels, nothing tuned on ground truth. Review clips are for debugging only.

## 1. Config

In [ ]:
import os, sys, json, math, glob, time, shutil, subprocess, datetime
from pathlib import Path

RUN_MODE   = os.environ.get("REF_RUN_MODE", "smoke")     # "selftest" | "smoke" | "full"
DET_MODEL  = os.environ.get("REF_DET_MODEL", "yolo11m.pt")          # person detector (COCO class 0)
POSE_MODEL = os.environ.get("REF_POSE_MODEL", "yolo11m-pose.pt")   # pose on referee crops only
DET_IMGSZ, DET_CONF, BATCH = 1280, 0.25, 16
POSE_IMGSZ, POSE_CONF = 640, 0.25
CROP_H     = 384          # referee crop is upscaled to this height
CROP_PAD_X = 0.9          # padding each side, in box widths (room for a stretched arm)
CROP_PAD_Y = 0.25         # padding top/bottom, in box heights
MIN_POSE_IOU = 0.3        # the pose person must overlap the detector box
SMOKE_RANGES = 8           # scan ranges processed in smoke mode, spread over the match

# referee identification (unsupervised torso-colour clusters)
REF_K               = 6       # colour clusters: 2 teams, 2 keepers, referees, other
REF_CALIB_FRAMES    = 600     # frames sampled across ranges to fit the clusters
REF_MAX_PER_FRAME   = 1.6     # referee cluster: mean members per frame where present (ref + an assistant ~1.2)
REF_MIN_SHARE       = 0.01    # ignore clusters with fewer than this share of detections
REF_TEAM_PER_FRAME  = 2.5     # clusters with more members per frame than this are teams
REF_MIN_SEP_FRAC    = 0.5     # candidates must be at least this fraction of the best colour separation from the teams
REF_DIST_Q          = 0.95    # a person is a referee if within this quantile of the cluster's own distances (x1.25)
REF_CLUSTER         = None    # set an int to force the cluster after looking at ref_clusters.png

# pointing gesture (angles in degrees, image plane)
KP_CONF        = 0.35
ARM_ELEV_MIN   = 70           # upper arm vs torso-down: 0 hanging, 90 horizontal, 180 straight up
ARM_EXT_MIN    = 140          # elbow angle: 180 = straight arm
HAND_ELEV_MIN  = 65           # shoulder->wrist vs torso-down
GAP_FRAMES     = 3
MIN_HOLD_S     = 0.4
SLOWMO_REL     = 0.35         # frame motion below this fraction of the median = slow motion
SLOWMO_EVENT_SHARE = 0.5      # events with more slow-mo frames than this are flagged replay
N_REVIEW_CLIPS = 24
SEED = 0

KAGGLE_INPUT, KAGGLE_WORKING = Path("/kaggle/input"), Path("/kaggle/working")
ON_KAGGLE = KAGGLE_INPUT.exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
KAGGLE_BATCH = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"
TQDM_MIN = 15.0 if KAGGLE_BATCH else 1.0
RESUME_GLOB = "/kaggle/input/*/referee/chunks"

def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "games.py").exists():
            return p
    return Path.cwd()

REPO = find_repo_root()
OUT_DIR = (KAGGLE_WORKING if ON_KAGGLE and KAGGLE_WORKING.exists() else REPO / "outputs") / "referee"
if RUN_MODE == "selftest":
    OUT_DIR = OUT_DIR / "selftest"
CHUNK_DIR = OUT_DIR / ("chunks" if RUN_MODE == "full" else f"chunks_{RUN_MODE}")
for d in (OUT_DIR, CHUNK_DIR, OUT_DIR / "clips"):
    d.mkdir(parents=True, exist_ok=True)

def find_file(name, extra_dirs=()):
    """A file by name: repo locations first, then anything under /kaggle/input."""
    for d in [*extra_dirs, REPO / "outputs" / "referee", REPO / "data" / "broadcast"]:
        p = Path(d) / name
        if p.exists():
            return p
    if KAGGLE_INPUT.exists():
        hit = next(KAGGLE_INPUT.rglob(name), None)
        if hit:
            return hit
    return None

assert RUN_MODE in ("selftest", "smoke", "full"), RUN_MODE
print(f"RUN_MODE={RUN_MODE}  det={DET_MODEL}  pose={POSE_MODEL}  kaggle={ON_KAGGLE}  out={OUT_DIR}")

## 2. Install and import

In [ ]:
if RUN_MODE != "selftest" and not os.environ.get("SS_SKIP_INSTALL"):
    try:
        import ultralytics  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=False)
for _p in ("tqdm", "scikit-learn"):
    try:
        __import__("sklearn" if _p == "scikit-learn" else _p)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _p], check=False)

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.cluster import KMeans

try:
    import torch
    CUDA = torch.cuda.is_available()
except ImportError:
    torch, CUDA = None, False
print("numpy", np.__version__, "| opencv", cv2.__version__, "| CUDA", CUDA)

## 3. Keypoint geometry: is an arm pointing?

COCO keypoints: 5/6 shoulders, 7/8 elbows, 9/10 wrists, 11/12 hips (left/right of the person).
All angles are measured in the image against the person's own torso axis (shoulder-mid → hip-mid),
so a tilted or leaning referee is handled.

In [ ]:
L_SH, R_SH, L_EL, R_EL, L_WR, R_WR, L_HIP, R_HIP = 5, 6, 7, 8, 9, 10, 11, 12
ARMS = {"L": (L_SH, L_EL, L_WR), "R": (R_SH, R_EL, R_WR)}

def _angle(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    if nu < 1e-6 or nv < 1e-6:
        return np.nan
    return float(np.degrees(np.arccos(np.clip(np.dot(u, v) / (nu * nv), -1.0, 1.0))))

def arm_features(kp, kc):
    """Per arm: elevation, extension, hand elevation, image direction, pointing flag. kp (17,2), kc (17,)."""
    hips = [k for k in (L_HIP, R_HIP) if kc[k] >= KP_CONF]
    shs = [k for k in (L_SH, R_SH) if kc[k] >= KP_CONF]
    out = {}
    if not hips or not shs:
        return out
    down = kp[hips].mean(0) - kp[shs].mean(0)               # torso axis, pointing down
    for side, (s, e, w) in ARMS.items():
        if min(kc[s], kc[e], kc[w]) < KP_CONF:
            continue
        upper, fore, reach = kp[e] - kp[s], kp[w] - kp[e], kp[w] - kp[s]
        elev = _angle(upper, down)
        ext = _angle(kp[s] - kp[e], kp[w] - kp[e])
        hand = _angle(reach, down)
        dx, dy = reach[0], -reach[1]                         # image y grows downwards
        ang = math.degrees(math.atan2(dy, dx))
        direction = "up" if 60 <= ang <= 120 else ("right" if -90 < ang < 60 else "left")
        pointing = bool(np.isfinite(elev) and elev >= ARM_ELEV_MIN and ext >= ARM_EXT_MIN and hand >= HAND_ELEV_MIN)
        out[side] = {"elev": elev, "ext": ext, "hand": hand, "dir_deg": ang, "direction": direction, "pointing": pointing}
    return out

# --- unit checks on hand-made skeletons (run in every mode) ---
def _skeleton(arm="down", side="R"):
    kp = np.zeros((17, 2)); kc = np.ones(17)
    kp[L_SH], kp[R_SH], kp[L_HIP], kp[R_HIP] = (90, 100), (110, 100), (92, 160), (108, 160)
    s = R_SH if side == "R" else L_SH; e, w = (R_EL, R_WR) if side == "R" else (L_EL, L_WR)
    sx = kp[s][0]; sgn = 1 if side == "R" else -1
    poses = {"down": ((sx, 125), (sx, 150)), "horizontal": ((sx + 25 * sgn, 100), (sx + 50 * sgn, 100)),
             "up": ((sx, 75), (sx, 50)), "run": ((sx + 5 * sgn, 125), (sx + 25 * sgn, 112)),
             "diag": ((sx + 18 * sgn, 82), (sx + 36 * sgn, 64))}
    kp[e], kp[w] = poses[arm]
    other = (L_EL, L_WR) if side == "R" else (R_EL, R_WR)
    os_ = L_SH if side == "R" else R_SH
    kp[other[0]], kp[other[1]] = (kp[os_][0], 125), (kp[os_][0], 150)
    return kp, kc

for arm, want, d in [("down", False, None), ("run", False, None), ("horizontal", True, "right"),
                     ("up", True, "up"), ("diag", True, "right")]:
    f = arm_features(*_skeleton(arm))["R"]
    assert f["pointing"] == want and (d is None or f["direction"] == d), (arm, f)
f = arm_features(*_skeleton("horizontal", side="L"))["L"]
assert f["pointing"] and f["direction"] == "left", f
print("arm geometry checks passed")

## 4. Torso colour features (for referee identification)

In [ ]:
def torso_feature(frame, kp, kc, box):
    """Lab mean/std + 8-bin hue histogram of the torso, pitch green removed. None if too few pixels."""
    h, w = frame.shape[:2]
    pts = [kp[k] for k in (L_SH, R_SH, R_HIP, L_HIP) if kc[k] >= KP_CONF]
    mask = np.zeros((h, w), np.uint8)
    if len(pts) >= 3:
        cv2.fillConvexPoly(mask, cv2.convexHull(np.round(pts).astype(np.int32)), 1)
    else:
        x0, y0, x1, y1 = box
        bw, bh = x1 - x0, y1 - y0
        mask[int(y0 + 0.2 * bh):int(y0 + 0.5 * bh), int(x0 + 0.3 * bw):int(x0 + 0.7 * bw)] = 1
    ys, xs = np.nonzero(mask)
    if len(ys) < 20:
        return None
    px = frame[ys, xs].reshape(-1, 1, 3)
    hsv = cv2.cvtColor(px, cv2.COLOR_BGR2HSV).reshape(-1, 3).astype(np.float32)
    keep = ~((hsv[:, 0] >= 35) & (hsv[:, 0] <= 85) & (hsv[:, 1] > 60))      # drop pitch green
    if keep.sum() < 15:
        return None
    lab = cv2.cvtColor(px[keep], cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    sat = hsv[keep, 1] > 50
    hist = np.histogram(hsv[keep][sat, 0], bins=8, range=(0, 180))[0].astype(np.float32)
    hist = hist / max(1.0, float(keep.sum()))
    return np.concatenate([lab.mean(0) / 255.0, lab.std(0) / 128.0, hist])

## 5. Models: person detector + pose on crops

Both work in **frame coordinates**: the detector returns boxes per frame; `CropPose` takes those
boxes, crops and upscales each, runs YOLO11-pose, keeps the pose person that best overlaps the box
and maps the keypoints back. Selftest swaps in fakes that return the known synthetic skeletons.

In [ ]:
def _resolve(weights):
    p = Path(weights)
    return str(p if p.exists() else (find_file(p.name) or weights))

def _iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0

class Detector:
    def __init__(self, weights=DET_MODEL):
        from ultralytics import YOLO
        self.model, self.device = YOLO(_resolve(weights)), (0 if CUDA else "cpu")
    def __call__(self, frames, frame_ids=None):
        res = self.model.predict(frames, imgsz=DET_IMGSZ, conf=DET_CONF, classes=[0], device=self.device, half=CUDA, verbose=False)
        return [(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()) if r.boxes is not None and len(r.boxes)
                else (np.zeros((0, 4)), np.zeros(0)) for r in res]

class CropPose:
    def __init__(self, weights=POSE_MODEL):
        from ultralytics import YOLO
        self.model, self.device = YOLO(_resolve(weights)), (0 if CUDA else "cpu")
    def __call__(self, frames, frame_ids, boxes_per_frame):
        """[[ (kp (17,2), kc (17,)) or None per box ] per frame], keypoints in frame coordinates."""
        crops, meta = [], []
        for fi, (fr, boxes) in enumerate(zip(frames, boxes_per_frame)):
            H, W = fr.shape[:2]
            for bi, (x0, y0, x1, y1) in enumerate(boxes):
                w, h = x1 - x0, y1 - y0
                X0, Y0 = int(max(0, x0 - CROP_PAD_X * w)), int(max(0, y0 - CROP_PAD_Y * h))
                X1, Y1 = int(min(W, x1 + CROP_PAD_X * w)), int(min(H, y1 + CROP_PAD_Y * h))
                if X1 - X0 < 4 or Y1 - Y0 < 4:
                    continue
                sc = CROP_H / float(Y1 - Y0)
                crops.append(cv2.resize(fr[Y0:Y1, X0:X1], None, fx=sc, fy=sc, interpolation=cv2.INTER_CUBIC))
                meta.append((fi, bi, X0, Y0, sc, ((x0 - X0) * sc, (y0 - Y0) * sc, (x1 - X0) * sc, (y1 - Y0) * sc)))
        out = [[None] * len(b) for b in boxes_per_frame]
        for k in range(0, len(crops), BATCH):
            res = self.model.predict(crops[k:k + BATCH], imgsz=POSE_IMGSZ, conf=POSE_CONF, device=self.device, half=CUDA, verbose=False)
            for (fi, bi, X0, Y0, sc, tb), r in zip(meta[k:k + BATCH], res):
                if r.boxes is None or not len(r.boxes) or r.keypoints is None:
                    continue
                ious = [_iou(bb, tb) for bb in r.boxes.xyxy.cpu().numpy()]
                j = int(np.argmax(ious))
                if ious[j] < MIN_POSE_IOU:
                    continue
                kp = r.keypoints.xy.cpu().numpy()[j] / sc + [X0, Y0]
                kc = r.keypoints.conf.cpu().numpy()[j] if r.keypoints.conf is not None else np.ones(17)
                out[fi][bi] = (kp, kc)
        return out

## 6. Inputs

In [ ]:
def _synthetic_inputs():
    """Selftest: a 40 s synthetic broadcast (960x540, 25 fps). 6 red + 6 white players and a yellow
    referee per frame; the referee points right in 8.0-9.6 s and up in 25.0-26.0 s; a red player raises
    a straight arm in 15-17 s (must NOT count); 30-34 s is slow motion (every frame shown 4x) with the
    referee pointing (event flagged replay)."""
    fps, W, H, n = 25, 960, 540, 1000
    rng = np.random.default_rng(SEED)
    base = rng.uniform([80, 120], [880, 480], size=(13, 2)); vel = rng.uniform(-40, 40, size=(13, 2))
    colours = [(40, 40, 200)] * 6 + [(235, 235, 235)] * 6 + [(30, 230, 230)]      # BGR: red, white, yellow
    truth = {}
    path = OUT_DIR / "synthetic_broadcast.mp4"
    wr = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    for f in range(n):
        t = f / fps
        tau = 30.0 + np.floor((t - 30.0) * fps / 4) / fps if 30.0 <= t < 34.0 else t
        img = np.zeros((H, W, 3), np.uint8); img[:] = (40, 130, 40)
        for k in range(0, W, 96):
            cv2.rectangle(img, (k, 0), (k + 48, H), (45, 145, 45), -1)
        objs = []
        for i in range(13):
            cx, cy = (base[i] + vel[i] * tau) % [W - 80, H - 120] + [40, 60]
            x0, y0 = cx - 15, cy - 45
            cv2.rectangle(img, (int(x0 + 4), int(y0 + 18)), (int(x0 + 26), int(y0 + 52)), colours[i], -1)
            cv2.circle(img, (int(cx), int(y0 + 10)), 8, (150, 180, 220), -1)
            cv2.rectangle(img, (int(x0 + 6), int(y0 + 52)), (int(x0 + 24), int(y0 + 88)), (20, 20, 20), -1)
            kp = np.zeros((17, 2)); kc = np.full(17, 0.9)
            kp[L_SH], kp[R_SH] = (x0 + 6, y0 + 20), (x0 + 24, y0 + 20)
            kp[L_HIP], kp[R_HIP] = (x0 + 8, y0 + 52), (x0 + 22, y0 + 52)
            arm = "down"
            if i == 12 and (8.0 <= tau < 9.6 or 30.5 <= tau < 33.5):
                arm = "right"
            elif i == 12 and 25.0 <= tau < 26.0:
                arm = "up"
            elif i == 0 and 15.0 <= tau < 17.0:
                arm = "up"
            sx, sy = kp[R_SH]
            if arm == "right":
                kp[R_EL], kp[R_WR] = (sx + 14, sy), (sx + 28, sy)
            elif arm == "up":
                kp[R_EL], kp[R_WR] = (sx, sy - 14), (sx, sy - 28)
            else:
                kp[R_EL], kp[R_WR] = (sx, sy + 14), (sx, sy + 28)
            kp[L_EL], kp[L_WR] = (kp[L_SH][0], kp[L_SH][1] + 14), (kp[L_SH][0], kp[L_SH][1] + 28)
            cv2.line(img, tuple(map(int, kp[R_SH])), tuple(map(int, kp[R_WR])), colours[i], 4)
            objs.append((np.array([x0, y0, x0 + 30, y0 + 90]), 0.9, kp, kc))
        truth[f] = objs
        wr.write(img)
    wr.release()
    ranges = pd.DataFrame({"range_id": [1, 2], "broadcast_first": [0, 500], "broadcast_last": [499, 999],
                           "frames": [500, 500], "window_ids": ["1", "2|3"]})
    windows = pd.DataFrame({"window_id": [1, 2, 3], "d": [1, 2, 3], "n": [1, 2, 3],
                            "kind": ["stoppage", "stoppage", "restart"], "direction": ["H->D", "A->D", "D->H"],
                            "half": [1, 1, 1], "anchor_tactical": [5200, 5700, 5950],
                            "broadcast_anchor": [200, 625, 750], "broadcast_first": [150, 575, 700],
                            "broadcast_last": [350, 775, 900], "view": ["good", "good", "good"]})
    return path, ranges, windows, truth


class FakeDetector:
    def __init__(self, truth):
        self.truth = truth
    def __call__(self, frames, frame_ids):
        return [(np.array([o[0] for o in self.truth.get(int(f), [])]).reshape(-1, 4),
                 np.array([o[1] for o in self.truth.get(int(f), [])])) for f in frame_ids]

class FakeCropPose:
    def __init__(self, truth):
        self.truth = truth
    def __call__(self, frames, frame_ids, boxes_per_frame):
        out = []
        for f, boxes in zip(frame_ids, boxes_per_frame):
            objs = self.truth.get(int(f), [])
            row = []
            for b in boxes:
                j = int(np.argmax([_iou(b, o[0]) for o in objs])) if objs else -1
                row.append((objs[j][2], objs[j][3]) if j >= 0 and _iou(b, objs[j][0]) > 0.5 else None)
            out.append(row)
        return out


if RUN_MODE == "selftest":
    VIDEO, RANGES, WINDOWS, _TRUTH = _synthetic_inputs()
    DETECT, POSE = FakeDetector(_TRUTH), FakeCropPose(_TRUTH)
else:
    VIDEO = find_file("BAR-ATH_broadcast.mp4")
    rp, wp = find_file("broadcast_scan_ranges.csv"), find_file("broadcast_windows.csv")
    missing = [n for n, p in (("BAR-ATH_broadcast.mp4", VIDEO), ("broadcast_scan_ranges.csv", rp),
                              ("broadcast_windows.csv", wp)) if p is None]
    if missing:
        raise FileNotFoundError(f"not found: {missing}. Add the broadcast video and the P2 outputs as Kaggle inputs.")
    RANGES, WINDOWS = pd.read_csv(rp), pd.read_csv(wp)
    if RUN_MODE == "smoke":
        RANGES = RANGES.iloc[np.linspace(0, len(RANGES) - 1, min(SMOKE_RANGES, len(RANGES))).astype(int)]
    DETECT, POSE = Detector(), CropPose()

cap = cv2.VideoCapture(str(VIDEO)); FPS = cap.get(cv2.CAP_PROP_FPS) or 25.0
NB = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
for d in glob.glob(RESUME_GLOB):
    for f in Path(d).glob("*.parquet"):
        if not (CHUNK_DIR / f.name).exists():
            shutil.copy2(f, CHUNK_DIR / f.name)
print(f"video {VIDEO.name}: {NB} frames @ {FPS} fps | ranges {len(RANGES)} ({int(RANGES['frames'].sum())} frames) | windows {len(WINDOWS)}")

## 7. Pass 1: people over the scan ranges

One parquet chunk per range (resumable): every person box with its torso-colour feature, plus a
per-frame motion value (for slow motion).

In [ ]:
def _thumb(fr):
    return cv2.resize(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY), (64, 36), interpolation=cv2.INTER_AREA).astype(np.float32)

def iter_range(a, b):
    """Yield (frame_id, frame) for broadcast frames a..b (exact seek, else read forward)."""
    cap = cv2.VideoCapture(str(VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, int(a))
    pos = int(round(cap.get(cv2.CAP_PROP_POS_FRAMES)))
    if pos != a:
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, a - 250)); pos = int(round(cap.get(cv2.CAP_PROP_POS_FRAMES)))
        while pos < a and cap.grab():
            pos += 1
    for f in range(a, b + 1):
        ok, fr = cap.read()
        if not ok:
            break
        yield f, fr
    cap.release()

def _batches(it, n):
    buf = []
    for x in it:
        buf.append(x)
        if len(buf) == n:
            yield buf; buf = []
    if buf:
        yield buf

_NOKP, _NOKC = np.zeros((17, 2)), np.zeros(17)

def detect_range(rid, a, b):
    path = CHUNK_DIR / f"people_{rid:04d}_{a}_{b}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    rows, prev = [], None
    for batch in _batches(iter_range(a, b), BATCH):
        ids, frames = [x[0] for x in batch], [x[1] for x in batch]
        for fid, fr, (boxes, confs) in zip(ids, frames, DETECT(frames, ids)):
            th = _thumb(fr); mot = np.nan if prev is None else float(np.abs(th - prev).mean()); prev = th
            if not len(boxes):
                rows.append({"range_id": rid, "frame": fid, "person": -1, "motion": mot}); continue
            for p, (bx, cf) in enumerate(zip(boxes, confs)):
                feat = torso_feature(fr, _NOKP, _NOKC, bx)
                rows.append({"range_id": rid, "frame": fid, "person": p, "motion": mot,
                             "x0": float(bx[0]), "y0": float(bx[1]), "x1": float(bx[2]), "y1": float(bx[3]),
                             "det_conf": float(cf), "feat": None if feat is None else feat.astype(np.float32).tolist()})
    df = pd.DataFrame(rows); df.to_parquet(path, index=False)
    return df

T0 = time.time()
parts = [detect_range(int(r.range_id), int(r.broadcast_first), int(r.broadcast_last))
         for r in tqdm(RANGES.itertuples(), total=len(RANGES), desc="pass 1: people", mininterval=TQDM_MIN)]
DET = pd.concat(parts, ignore_index=True)
PERS = DET[DET["person"] >= 0].reset_index(drop=True)
print(f"{DET['frame'].nunique()} frames, {len(PERS)} person boxes, {PERS.groupby('frame').size().mean():.1f} per frame "
      f"| median box height {float((PERS['y1'] - PERS['y0']).median()):.0f}px | {time.time() - T0:.0f}s")

## 8. Find the referee cluster

k-means on the torso features of a calibration sample. Per cluster: share of detections, mean members
per frame where present, and coverage (share of ranges where it appears). The referee is the cluster
with the highest coverage among those with ≤ `REF_MAX_PER_FRAME` members per frame.

In [ ]:
has = PERS["feat"].notna()
X_all = np.stack(PERS.loc[has, "feat"].to_numpy())
calib_frames = PERS.loc[has, "frame"].drop_duplicates().sample(
    min(REF_CALIB_FRAMES, PERS.loc[has, "frame"].nunique()), random_state=SEED)
cal = PERS.loc[has & PERS["frame"].isin(calib_frames)]
Xc = np.stack(cal["feat"].to_numpy())
mu, sd = Xc.mean(0), Xc.std(0) + 1e-6
km = KMeans(n_clusters=min(REF_K, len(Xc)), n_init=10, random_state=SEED).fit((Xc - mu) / sd)
lab_all = km.predict((X_all - mu) / sd)
dist_all = np.linalg.norm((X_all - mu) / sd - km.cluster_centers_[lab_all], axis=1)
PERS["cluster"] = -1; PERS["cdist"] = np.nan
PERS.loc[has, "cluster"] = lab_all; PERS.loc[has, "cdist"] = dist_all

stats = []
for c in range(km.n_clusters):
    m = PERS[PERS["cluster"] == c]
    stats.append({"cluster": c, "share": len(m) / max(1, has.sum()),
                  "per_frame": m.groupby("frame").size().mean() if len(m) else np.nan,
                  "coverage": m["range_id"].nunique() / max(1, PERS["range_id"].nunique()),
                  "lab_L": float(np.mean([f[0] for f in m["feat"]])) * 255 if len(m) else np.nan})
STATS = pd.DataFrame(stats)
C = km.cluster_centers_
teams = STATS.loc[STATS["per_frame"] > REF_TEAM_PER_FRAME, "cluster"].astype(int).tolist()
if not teams:                                             # e.g. only close-ups: the two biggest clusters
    teams = STATS.sort_values("share").tail(2)["cluster"].astype(int).tolist()
STATS["separation"] = [float(min(np.linalg.norm(C[c] - C[t]) for t in teams)) if c not in teams else 0.0
                       for c in STATS["cluster"].astype(int)]
cand = STATS[(STATS["per_frame"] <= REF_MAX_PER_FRAME) & (STATS["share"] >= REF_MIN_SHARE)]
if len(cand):
    cand = cand[cand["separation"] >= REF_MIN_SEP_FRAC * cand["separation"].max()]
# kit contrasts with both teams (separation); then on screen in most ranges (keepers only near their goal)
REF_C = int(REF_CLUSTER) if REF_CLUSTER is not None else (
    int(cand.sort_values(["coverage", "share"]).iloc[-1]["cluster"]) if len(cand) else None)
print(STATS.round(3).to_string(index=False))
print("team clusters:", teams, "| referee cluster:", REF_C, "(forced)" if REF_CLUSTER is not None else "(auto)")
if REF_C is None:
    raise RuntimeError("no cluster looks like a referee. Look at ref_clusters.png and set REF_CLUSTER.")
# the referee kit can split (sun / shade): join clusters closer to it than half its separation, never a team
sep_ref = float(STATS.set_index("cluster").loc[REF_C, "separation"])
REF_SET = [REF_C] + [int(c) for c in STATS["cluster"].astype(int)
                     if c != REF_C and c not in teams and np.linalg.norm(C[c] - C[REF_C]) < 0.5 * sep_ref]
PERS["is_ref"] = False
for c in REF_SET:
    own = PERS.loc[PERS["cluster"] == c, "cdist"]
    PERS.loc[(PERS["cluster"] == c) & (PERS["cdist"] <= float(own.quantile(REF_DIST_Q) * 1.25)), "is_ref"] = True
print("referee clusters:", REF_SET)

# montage: 8 crops per cluster
def _crop(row):
    cap = cv2.VideoCapture(str(VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame)); ok, fr = cap.read(); cap.release()
    if not ok:
        return np.zeros((96, 48, 3), np.uint8)
    x0, y0, x1, y1 = (int(max(0, v)) for v in (row.x0, row.y0, row.x1, row.y1))
    c = fr[y0:max(y0 + 2, y1), x0:max(x0 + 2, x1)]
    return cv2.cvtColor(cv2.resize(c, (48, 96)), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(km.n_clusters, 8, figsize=(8, 1.6 * km.n_clusters))
for c in range(km.n_clusters):
    m = PERS[PERS["cluster"] == c]
    sample = m.sample(min(8, len(m)), random_state=SEED) if len(m) else m
    for j in range(8):
        ax = axes[c, j]; ax.axis("off")
        if j < len(sample):
            ax.imshow(_crop(sample.iloc[j]))
    s = STATS.iloc[c]
    axes[c, 0].set_title(f"c{c}{' REF' if c in REF_SET else ''} share {s['share']:.2f} pf {s['per_frame']:.1f}", fontsize=7, loc="left")
plt.tight_layout(); plt.savefig(OUT_DIR / "ref_clusters.png", dpi=110); plt.show()
print(f"referees: {int(PERS['is_ref'].sum())} detections on {PERS.loc[PERS['is_ref'], 'frame'].nunique()} frames")

## 8b. Pass 2: pose on the referee crops

Only frames that contain a referee box are decoded again; each referee is cropped, upscaled and
posed, and the arm features are computed from its keypoints.

In [ ]:
def pose_range(rid, a, b, refs):
    path = CHUNK_DIR / f"pose_{rid:04d}_{a}_{b}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    by_frame = {f: g for f, g in refs.groupby("frame")}
    rows = []
    wanted = (x for x in iter_range(a, b) if x[0] in by_frame)
    for batch in _batches(wanted, BATCH):
        ids, frames = [x[0] for x in batch], [x[1] for x in batch]
        boxes = [by_frame[f][["x0", "y0", "x1", "y1"]].to_numpy() for f in ids]
        for fid, bx, persons, poses in zip(ids, boxes, [by_frame[f]["person"].to_numpy() for f in ids], POSE(frames, ids, boxes)):
            for p, box, kpkc in zip(persons, bx, poses):
                r = {"range_id": rid, "frame": fid, "person": int(p), "posed": kpkc is not None}
                if kpkc is not None:
                    kp, kc = kpkc
                    arms = arm_features(kp, kc)
                    r["kp"] = np.round(kp, 1).astype(np.float32).ravel().tolist()
                    r["kc"] = np.round(kc, 3).astype(np.float32).tolist()
                    for side in ("L", "R"):
                        a_ = arms.get(side)
                        r[f"{side}_elev"] = a_["elev"] if a_ else np.nan
                        r[f"{side}_ext"] = a_["ext"] if a_ else np.nan
                        r[f"{side}_hand"] = a_["hand"] if a_ else np.nan
                        r[f"{side}_dir"] = a_["direction"] if a_ else ""
                        r[f"{side}_point"] = bool(a_["pointing"]) if a_ else False
                rows.append(r)
    df = pd.DataFrame(rows)
    for side in ("L", "R"):
        for c, v in ((f"{side}_point", False), (f"{side}_dir", ""), (f"{side}_elev", np.nan), (f"{side}_ext", np.nan), (f"{side}_hand", np.nan)):
            if c not in df:
                df[c] = v
    for c in ("kp", "kc"):
        if c not in df:
            df[c] = None
    df.to_parquet(path, index=False)
    return df

refs_all = PERS[PERS["is_ref"]]
pparts = [pose_range(int(r.range_id), int(r.broadcast_first), int(r.broadcast_last), refs_all[refs_all["range_id"] == r.range_id])
          for r in tqdm(RANGES.itertuples(), total=len(RANGES), desc="pass 2: referee pose", mininterval=TQDM_MIN)
          if (refs_all["range_id"] == r.range_id).any()]
POSED = pd.concat(pparts, ignore_index=True) if pparts else pd.DataFrame(columns=["range_id", "frame", "person", "posed"])
POSED["L_point"] = POSED["L_point"].fillna(False).astype(bool); POSED["R_point"] = POSED["R_point"].fillna(False).astype(bool)
print(f"referee boxes posed: {int(POSED['posed'].sum())}/{len(POSED)} ({POSED['posed'].mean():.0%}) | {time.time() - T0:.0f}s")

## 9. Gesture events

In [ ]:
REF = PERS[PERS["is_ref"]].drop(columns=["feat"]).merge(POSED, on=["range_id", "frame", "person"], how="inner")
REF = REF[REF["posed"]].copy()
REF["pointing"] = REF["L_point"] | REF["R_point"]
REF["arm"] = np.where(REF["R_point"], "R", np.where(REF["L_point"], "L", ""))
REF["direction"] = np.where(REF["R_point"], REF["R_dir"], np.where(REF["L_point"], REF["L_dir"], ""))
REF["box_h"] = REF["y1"] - REF["y0"]

frames_all = DET.groupby("frame").agg(range_id=("range_id", "first"), motion=("motion", "first")).reset_index()
med_motion = float(np.nanmedian(frames_all["motion"]))
frames_all["slowmo"] = frames_all["motion"] < SLOWMO_REL * med_motion
point_frames = REF[REF["pointing"]].groupby("frame").agg(
    arm=("arm", lambda s: s.mode().iat[0]), direction=("direction", lambda s: s.mode().iat[0]),
    elev=("R_elev", "max"), box_h=("box_h", "max"), det_conf=("det_conf", "max")).reset_index()
pf = frames_all.merge(point_frames, on="frame", how="left")
pf["pointing"] = pf["arm"].notna()
ref_visible = set(REF["frame"])

# windows: broadcast span + linear map to tactical (both 25 fps, live): t = anchor_t + (b - anchor_b)
WIN = WINDOWS.copy()
def windows_for(b):
    m = WIN[(WIN["broadcast_first"] <= b) & (WIN["broadcast_last"] >= b)]
    return m

events = []
for rid, g in pf.sort_values("frame").groupby("range_id"):
    fr, pt = g["frame"].to_numpy(), g["pointing"].to_numpy()
    idx = np.flatnonzero(pt)
    if not len(idx):
        continue
    runs, s = [], idx[0]
    for i, j in zip(idx[:-1], idx[1:]):
        if fr[j] - fr[i] > GAP_FRAMES + 1:
            runs.append((s, i)); s = j
    runs.append((s, idx[-1]))
    for s, e in runs:
        a, b = int(fr[s]), int(fr[e])
        dur = (b - a + 1) / FPS
        if dur < MIN_HOLD_S:
            continue
        seg = g[(g["frame"] >= a) & (g["frame"] <= b)]
        on = seg[seg["pointing"]]
        mid = (a + b) // 2
        ws = windows_for(mid)
        w0 = ws.iloc[(ws["broadcast_anchor"] - mid).abs().argsort()].iloc[0] if len(ws) else None
        slow = float(seg["slowmo"].mean())
        events.append({
            "event_id": len(events) + 1, "range_id": int(rid), "broadcast_first": a, "broadcast_last": b,
            "seconds": round(dur, 2), "pointing_share": round(len(on) / len(seg), 3),
            "arm": on["arm"].mode().iat[0], "direction": on["direction"].mode().iat[0],
            "max_elev": round(float(on["elev"].max()), 1), "ref_box_h": round(float(on["box_h"].median()), 1),
            "slowmo_share": round(slow, 3), "replay_flag": bool(slow >= SLOWMO_EVENT_SHARE),
            "window_ids": "|".join(map(str, ws["window_id"].tolist())),
            "d": int(w0["d"]) if w0 is not None else -1, "kind": w0["kind"] if w0 is not None else "",
            "offset_from_anchor_s": round((a - int(w0["broadcast_anchor"])) / FPS, 2) if w0 is not None else np.nan,
            "tactical_first": int(w0["anchor_tactical"] + (a - w0["broadcast_anchor"])) if w0 is not None else -1,
            "tactical_last": int(w0["anchor_tactical"] + (b - w0["broadcast_anchor"])) if w0 is not None else -1,
        })
EVENT_COLS = ["event_id", "range_id", "broadcast_first", "broadcast_last", "seconds", "pointing_share", "arm",
              "direction", "max_elev", "ref_box_h", "slowmo_share", "replay_flag", "window_ids", "d", "kind",
              "offset_from_anchor_s", "tactical_first", "tactical_last"]
EVENTS = pd.DataFrame(events, columns=EVENT_COLS)
print(f"{len(EVENTS)} pointing events ({int(EVENTS['replay_flag'].sum()) if len(EVENTS) else 0} flagged slow-motion)")
if len(EVENTS):
    print(EVENTS.groupby(["kind", "direction"]).size().to_string())

## 10. Per window and per v6 stretch

In [ ]:
scanned = set(DET["frame"])
wrows = []
for w in WIN.itertuples():
    fr = np.arange(int(w.broadcast_first), int(w.broadcast_last) + 1) if w.broadcast_first >= 0 else np.array([], int)
    sc = [f for f in fr if f in scanned]
    ev = EVENTS[EVENTS["window_ids"].str.split("|").apply(lambda ids: str(w.window_id) in ids)] if len(EVENTS) else EVENTS
    live_ev = ev[~ev["replay_flag"]] if len(ev) else ev
    wrows.append({"window_id": int(w.window_id), "d": int(w.d), "kind": w.kind, "view": getattr(w, "view", ""),
                  "frames_scanned": len(sc),
                  "ref_visible_share": round(np.mean([f in ref_visible for f in sc]), 3) if sc else np.nan,
                  "events": int(len(live_ev)), "events_replay": int(len(ev) - len(live_ev)),
                  "gesture": bool(len(live_ev)),
                  "first_event_offset_s": float(live_ev["offset_from_anchor_s"].min()) if len(live_ev) else np.nan,
                  "directions": "|".join(sorted(set(live_ev["direction"]))) if len(live_ev) else ""})
WROWS = pd.DataFrame(wrows)
WROWS["status"] = np.where(WROWS["frames_scanned"] == 0, "not_scanned",
                  np.where(WROWS["gesture"], "gesture",
                  np.where(WROWS["ref_visible_share"].fillna(0) < 0.1, "referee_not_visible", "no_gesture")))
SROWS = (WROWS.groupby("d").agg(stoppage_status=("status", lambda s: "|".join(s[WROWS.loc[s.index, "kind"] == "stoppage"])),
                                restart_status=("status", lambda s: "|".join(s[WROWS.loc[s.index, "kind"] == "restart"])),
                                gesture=("gesture", "any"), events=("events", "sum")).reset_index())
SROWS["referee_signal"] = np.where(SROWS["gesture"], "yes",
                          np.where((SROWS["stoppage_status"] + SROWS["restart_status"]).str.contains("no_gesture"), "no", "unknown"))
print(WROWS["status"].value_counts().to_string())
print("v6 stretches:", SROWS["referee_signal"].value_counts().to_dict())

## 11. Review clips (skeleton overlay, debugging only)

In [ ]:
SKEL = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12)]
def write_clip(ev, path, pad=12):
    a, b = int(ev.broadcast_first) - pad, int(ev.broadcast_last) + pad
    cap = cv2.VideoCapture(str(VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, a))
    wr = None
    sub = REF[(REF["frame"] >= a) & (REF["frame"] <= b)]
    for f in range(max(0, a), b + 1):
        ok, fr = cap.read()
        if not ok:
            break
        for r in sub[sub["frame"] == f].itertuples():
            kp = np.asarray(r.kp).reshape(17, 2)
            col = (0, 0, 255) if r.pointing else (0, 255, 255)
            cv2.rectangle(fr, (int(r.x0), int(r.y0)), (int(r.x1), int(r.y1)), col, 2)
            for i, j in SKEL:
                cv2.line(fr, tuple(map(int, kp[i])), tuple(map(int, kp[j])), col, 2)
        cv2.putText(fr, f"event {ev.event_id} d={ev.d} {ev.kind} {ev.direction} frame {f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        fr = cv2.resize(fr, (fr.shape[1] // 2, fr.shape[0] // 2))
        if wr is None:
            wr = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (fr.shape[1], fr.shape[0]))
        wr.write(fr)
    cap.release()
    if wr is not None:
        wr.release()

if len(EVENTS):
    pick = EVENTS.sample(min(N_REVIEW_CLIPS, len(EVENTS)), random_state=SEED)
    for ev in tqdm(pick.itertuples(), total=len(pick), desc="review clips", mininterval=TQDM_MIN):
        write_clip(ev, OUT_DIR / "clips" / f"event_{ev.event_id:04d}_d{ev.d}_{ev.kind}.mp4")
print("clips in", OUT_DIR / "clips")

## 12. Checks and outputs

In [ ]:
CHECKS = []
def check(name, ok, detail="", critical=True):
    CHECKS.append({"name": name, "passed": bool(ok), "critical": critical, "detail": str(detail)})

check("referee boxes posed (>= 50%)", len(POSED) > 0 and POSED["posed"].mean() >= 0.5,
      f"{POSED['posed'].mean():.0%}" if len(POSED) else "none", critical=False)
check("detector ran on every scan range", DET["range_id"].nunique() == len(RANGES), f"{DET['range_id'].nunique()}/{len(RANGES)}")
check("people detected", len(PERS) > 0, len(PERS))
rs = STATS.set_index("cluster").loc[REF_C]
check("referee cluster ~1 per frame", rs["per_frame"] <= REF_MAX_PER_FRAME, f"{rs['per_frame']:.2f}")
check("referee visible in most scanned frames", PERS.loc[PERS['is_ref'], 'frame'].nunique() / max(1, DET['frame'].nunique()) >= 0.3,
      f"{PERS.loc[PERS['is_ref'], 'frame'].nunique() / max(1, DET['frame'].nunique()):.0%}", critical=False)
check("at least one pointing event", len(EVENTS) > 0, len(EVENTS), critical=RUN_MODE != "smoke")

if RUN_MODE == "selftest":
    ev = EVENTS[~EVENTS["replay_flag"]] if len(EVENTS) else EVENTS
    def has(a, b, direction):
        return bool(len(ev[(ev["broadcast_first"] <= b) & (ev["broadcast_last"] >= a) & (ev["direction"] == direction)]))
    check("selftest: referee cluster is the yellow kit", PERS.loc[PERS["is_ref"], "x0"].count() > 0 and
          (PERS.loc[PERS["is_ref"], "feat"].apply(lambda f: f[2] * 255).mean() > 150), "Lab b of referees")
    check("selftest: pointing right at 8.0-9.6 s", has(200, 240, "right"))
    check("selftest: pointing up at 25-26 s", has(625, 650, "up"))
    check("selftest: player raising an arm (15-17 s) is not an event", not len(EVENTS[(EVENTS["broadcast_first"] <= 425) & (EVENTS["broadcast_last"] >= 375)]))
    check("selftest: slow-motion pointing (30-34 s) is flagged replay",
          bool(len(EVENTS)) and EVENTS[(EVENTS["broadcast_first"] >= 750) & (EVENTS["broadcast_first"] < 850)]["replay_flag"].all()
          and len(EVENTS[(EVENTS["broadcast_first"] >= 750) & (EVENTS["broadcast_first"] < 850)]) > 0)
    check("selftest: event timing within 2 frames", has(198, 202, "right") and
          abs(int(ev[ev["direction"] == "right"]["broadcast_first"].min()) - 200) <= 2)
    check("selftest: stretch 1 and 2 have a referee signal",
          set(SROWS.loc[SROWS["referee_signal"] == "yes", "d"]) >= {1, 2}, SROWS[["d", "referee_signal"]].to_dict("records"))

EVENTS.to_csv(OUT_DIR / "referee_events.csv", index=False)
WROWS.to_csv(OUT_DIR / "referee_windows.csv", index=False)
SROWS.to_csv(OUT_DIR / "referee_stretches.csv", index=False)
REF.to_parquet(OUT_DIR / "referee_frames.parquet", index=False)
summary = {
    "run_mode": RUN_MODE, "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "models": {"detector": DET_MODEL, "pose": POSE_MODEL} if RUN_MODE != "selftest" else "fakes (selftest)",
    "config": {k: globals()[k] for k in ("DET_IMGSZ", "DET_CONF", "POSE_IMGSZ", "CROP_H", "CROP_PAD_X", "REF_K", "REF_MAX_PER_FRAME", "KP_CONF", "ARM_ELEV_MIN",
                                         "ARM_EXT_MIN", "HAND_ELEV_MIN", "GAP_FRAMES", "MIN_HOLD_S", "SLOWMO_REL")},
    "frames_scanned": int(DET["frame"].nunique()), "person_detections": int(len(PERS)),
    "referee_cluster": REF_C, "referee_clusters": REF_SET, "team_clusters": teams, "cluster_stats": STATS.round(4).to_dict("records"),
    "events": int(len(EVENTS)), "events_replay_flagged": int(EVENTS["replay_flag"].sum()) if len(EVENTS) else 0,
    "window_status": WROWS["status"].value_counts().to_dict(),
    "stretch_referee_signal": SROWS["referee_signal"].value_counts().to_dict(),
    "runtime_sec": round(time.time() - T0, 1), "checks": CHECKS,
}
with open(OUT_DIR / "referee_summary.json", "w") as fh:
    json.dump(summary, fh, indent=1, default=float)
chk = pd.DataFrame(CHECKS); print(chk.to_string(index=False))
n_fail = int((~chk["passed"] & chk["critical"]).sum())
print(f"\n{int(chk['passed'].sum())}/{len(chk)} checks passed, {n_fail} critical failure(s) | outputs in {OUT_DIR}")
if RUN_MODE == "selftest":
    assert n_fail == 0, "selftest failed"
    print("SELFTEST PASSED")